# OIBSIP — Data Analytics Level 2, Task 4
## Unveiling the Android App Market (Google Play Store Analysis)

**Objective:** Clean and analyze Google Play Store apps and reviews; explore categories, ratings, installs, pricing, and review sentiment.

> Included CSVs are **small demo data** so the notebook runs immediately. Replace them with full Kaggle datasets for genuine findings. Never report demo results as real market statistics.

## 1. Setup and data sources
Download **Google Play Store Apps** and **Google Play Store User Reviews** from:
https://www.kaggle.com/datasets/lava18/google-play-store-apps
Place them in `data/` as `googleplaystore.csv` and `googleplaystore_user_reviews.csv`. The notebook automatically uses full datasets if present; otherwise it loads demo CSVs.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA=Path("data")
apps_path=DATA/"googleplaystore.csv"
reviews_path=DATA/"googleplaystore_user_reviews.csv"
if not apps_path.exists(): apps_path=DATA/"googleplay_apps_demo.csv"
if not reviews_path.exists(): reviews_path=DATA/"googleplay_reviews_demo.csv"
apps=pd.read_csv(apps_path); reviews=pd.read_csv(reviews_path)
print("Apps:",apps.shape,"Reviews:",reviews.shape)
display(apps.head()); display(reviews.head())

## 2. Inspection and data cleaning

In [ ]:
apps.columns=[c.strip() for c in apps.columns]
reviews.columns=[c.strip() for c in reviews.columns]
print(apps.info()); display(apps.isna().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:",apps.duplicated().sum()); apps=apps.drop_duplicates().copy()

def parse_installs(v):
    if pd.isna(v): return np.nan
    s=re.sub(r"[^0-9]","",str(v)); return float(s) if s else np.nan
def parse_size_mb(v):
    if pd.isna(v) or str(v).lower().strip()=="varies with device": return np.nan
    m=re.match(r"([0-9.]+)\s*([kmg]?)",str(v).strip(),re.I)
    if not m: return np.nan
    n,u=float(m.group(1)),m.group(2).lower()
    return n/1024 if u=="k" else n*1024 if u=="g" else n
if "Installs" in apps: apps["Installs_num"]=apps["Installs"].apply(parse_installs)
if "Size" in apps: apps["Size_MB"]=apps["Size"].apply(parse_size_mb)
if "Price" in apps: apps["Price_num"]=pd.to_numeric(apps["Price"].astype(str).str.replace("$","",regex=False),errors="coerce")
if "Rating" in apps: apps["Rating_num"]=pd.to_numeric(apps["Rating"],errors="coerce")
if "Reviews" in apps: apps["Reviews_num"]=pd.to_numeric(apps["Reviews"],errors="coerce")
display(apps.head())

## 3. Category distribution / saturation

In [ ]:
if "Category" in apps:
    counts=apps["Category"].value_counts().head(15)
    plt.figure(figsize=(12,6)); sns.barplot(x=counts.values,y=counts.index)
    plt.title("Top 15 Categories by App Count"); plt.xlabel("Number of apps"); plt.tight_layout(); plt.show()
    display(counts.to_frame("App_Count"))

## 4. Ratings analysis

In [ ]:
if "Rating_num" in apps:
    sns.histplot(apps["Rating_num"].dropna(),bins=20,kde=True)
    plt.title("Distribution of App Ratings"); plt.tight_layout(); plt.show()
if {"Category","Rating_num"}.issubset(apps.columns):
    avg=apps.groupby("Category")["Rating_num"].mean().sort_values(ascending=False)
    display(avg.head(15).to_frame("Average_Rating"))
    plt.figure(figsize=(12,6)); sns.barplot(x=avg.head(15).values,y=avg.head(15).index)
    plt.title("Top Categories by Average Rating"); plt.tight_layout(); plt.show()

## 5. App size vs installs

In [ ]:
if {"Size_MB","Installs_num"}.issubset(apps.columns):
    d=apps.dropna(subset=["Size_MB","Installs_num"])
    sns.scatterplot(data=d,x="Size_MB",y="Installs_num",alpha=.6)
    plt.yscale("log"); plt.title("App Size vs Installs (log scale)"); plt.tight_layout(); plt.show()
    print("Pearson correlation:",d[["Size_MB","Installs_num"]].corr().iloc[0,1])

## 6. Pricing analysis
Install × price is only a rough estimate, not actual revenue.

In [ ]:
if "Type" in apps:
    display(apps["Type"].value_counts(dropna=False).to_frame("App_Count"))
    apps["Type"].value_counts().plot(kind="bar"); plt.title("Free vs Paid Apps"); plt.tight_layout(); plt.show()
if {"Price_num","Type"}.issubset(apps.columns):
    paid=apps[apps["Type"].astype(str).str.lower()=="paid"]
    display(paid["Price_num"].describe())
if {"Price_num","Installs_num","Category"}.issubset(apps.columns):
    apps["Estimated_Revenue"]=apps["Price_num"].fillna(0)*apps["Installs_num"].fillna(0)
    rev=apps.groupby("Category")["Estimated_Revenue"].sum().sort_values(ascending=False)
    display(rev.head(10).to_frame("Estimated_Revenue"))
    plt.figure(figsize=(11,5)); sns.barplot(x=rev.head(10).values,y=rev.head(10).index)
    plt.title("Top Categories by Estimated Paid-App Revenue"); plt.tight_layout(); plt.show()

## 7. Review sentiment with TextBlob
Polarity >0 = positive, <0 = negative, and 0 = neutral. This baseline may miss sarcasm/context.

In [ ]:
from textblob import TextBlob
review_col="Translated_Review" if "Translated_Review" in reviews.columns else ("Review" if "Review" in reviews.columns else None)
if review_col:
    reviews["Polarity"]=reviews[review_col].fillna("").astype(str).apply(lambda x:TextBlob(x).sentiment.polarity)
    reviews["Sentiment_Label"]=np.select([reviews["Polarity"]>0,reviews["Polarity"]<0],["Positive","Negative"],default="Neutral")
    display(reviews["Sentiment_Label"].value_counts().to_frame("Review_Count"))
    sns.countplot(data=reviews,x="Sentiment_Label",order=["Positive","Neutral","Negative"])
    plt.title("Review Sentiment"); plt.tight_layout(); plt.show()
else: print("Review text column not found.")

## 8. Sentiment by category

In [ ]:
if {"App","Sentiment_Label"}.issubset(reviews.columns) and {"App","Category"}.issubset(apps.columns):
    app_cat=apps[["App","Category"]].drop_duplicates()
    rc=reviews.merge(app_cat,on="App",how="left")
    tab=pd.crosstab(rc["Category"],rc["Sentiment_Label"],normalize="index")
    display(tab.sort_values("Positive",ascending=False).head(15))
    tab.sort_values("Positive",ascending=False).head(15).plot(kind="bar",stacked=True,figsize=(12,6))
    plt.title("Sentiment Share by Category"); plt.ylabel("Share"); plt.tight_layout(); plt.show()

## 9. Optional interactive Plotly

In [ ]:
try:
    import plotly.express as px
    if {"Category","Installs_num"}.issubset(apps.columns):
        d=apps.groupby("Category",as_index=False)["Installs_num"].median().sort_values("Installs_num",ascending=False).head(20)
        px.bar(d,x="Category",y="Installs_num",title="Median Installs by Category").show()
except ImportError: print("Install plotly: pip install plotly")

## 10. Three data-driven insights (fill after running on full data)
1. **Category opportunity:** compare category app counts with ratings/installs to identify less-saturated niches.
2. **User experience:** identify categories with negative sentiment and investigate recurring review complaints.
3. **Monetization:** compare paid/free shares and rough paid-app revenue by category.

Replace these with three numeric observations from your full dataset.

## 11. Limitations
- Dataset snapshot may be old, not a live Google Play census.
- Missing values and install buckets can bias results.
- Price × installs is only a rough revenue proxy.
- TextBlob can miss sarcasm and context.
- Demo data is illustrative only.

## References
- Kaggle Google Play Store Apps: https://www.kaggle.com/datasets/lava18/google-play-store-apps
- pandas: https://pandas.pydata.org/docs/
- seaborn: https://seaborn.pydata.org/
- TextBlob: https://textblob.readthedocs.io/
- Plotly: https://plotly.com/python/